# Notebook 1: Report Structure and Metadata Audit

## Purpose
Parse all 17 Markdown sustainability reports and audit their structural quality:
- Document structure (headings, tables, figures)
- Metadata quality (page markers, company info)
- Element distribution across reports
- Heading hierarchy consistency

## Outputs
- Summary report of document statistics
- CSV artifact: `data/artifacts/report_audit_summary.csv`

In [ ]:
import sys
from pathlib import Path

# Add project root to path for imports
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import json
from collections import defaultdict

from rag.config import REPORTS_MD_DIR, COMPANY_MAP, ARTIFACTS_DIR
from rag.parser import parse_all_reports, parse_report

print(f"Project root: {project_root}")
print(f"Reports directory: {REPORTS_MD_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")

In [ ]:
# Control smoke test mode
SMOKE_TEST = True

if SMOKE_TEST:
    print("SMOKE TEST MODE: Processing first 2 companies only")
else:
    print("FULL MODE: Processing all 17 companies")

In [ ]:
# Parse all reports
print("Parsing all reports...")
all_parsed = parse_all_reports()

if SMOKE_TEST:
    companies_to_process = list(all_parsed.keys())[:2]
    reports = {k: all_parsed[k] for k in companies_to_process}
else:
    reports = all_parsed

print(f"Parsed {len(reports)} reports")

# Create summary table
summary_rows = []
for company, parsed_report in reports.items():
    summary_rows.append({
        'Company': company,
        'Display Name': COMPANY_MAP.get(company, company),
        'Elements': len(parsed_report.elements),
        'Total Pages': parsed_report.total_pages,
        'Source File': Path(parsed_report.source_file).name if parsed_report.source_file else 'N/A',
    })

summary_df = pd.DataFrame(summary_rows)
print("\nReport Summary:")
print(summary_df.to_string(index=False))

## Element Distribution Analysis

In [ ]:
# Count element types per report
element_stats = []

for company, parsed_report in reports.items():
    type_counts = defaultdict(int)
    for elem in parsed_report.elements:
        type_counts[elem.element_type] += 1
    
    row = {'Company': company}
    row.update(dict(type_counts))
    element_stats.append(row)

element_df = pd.DataFrame(element_stats).fillna(0)
element_df = element_df.astype({col: int for col in element_df.columns if col != 'Company'})
print("Element Type Distribution:")
print(element_df.to_string(index=False))

## Page Marker Coverage

In [ ]:
# Check % of elements with page metadata per report
page_coverage = []

for company, parsed_report in reports.items():
    total_elems = len(parsed_report.elements)
    with_page_info = sum(1 for e in parsed_report.elements if e.page_number >= 0)
    coverage_pct = (with_page_info / total_elems * 100) if total_elems > 0 else 0
    
    page_coverage.append({
        'Company': company,
        'Total Elements': total_elems,
        'With Page Info': with_page_info,
        'Coverage %': f"{coverage_pct:.1f}%"
    })

page_df = pd.DataFrame(page_coverage)
print("Page Marker Coverage:")
print(page_df.to_string(index=False))

## Heading Hierarchy Depth

In [ ]:
# Analyze heading levels distribution
heading_stats = []

for company, parsed_report in reports.items():
    heading_levels = defaultdict(int)
    max_level = 0
    
    for elem in parsed_report.elements:
        if elem.element_type == 'heading' and elem.heading_level > 0:
            heading_levels[f"H{elem.heading_level}"] += 1
            max_level = max(max_level, elem.heading_level)
    
    row = {'Company': company, 'Max Heading Level': max_level}
    row.update({f"H{i}": heading_levels[f"H{i}"] for i in range(1, 7)})
    heading_stats.append(row)

heading_df = pd.DataFrame(heading_stats).fillna(0)
heading_df = heading_df.astype({col: int for col in heading_df.columns if col != 'Company'})
print("Heading Hierarchy Distribution:")
print(heading_df.to_string(index=False))

## Table and Figure Audit

In [ ]:
# Count tables and figures per report
table_figure_stats = []

for company, parsed_report in reports.items():
    tables = sum(1 for e in parsed_report.elements if e.element_type == 'table')
    figures = sum(1 for e in parsed_report.elements if e.element_type == 'figure')
    
    # Check HTML table integrity (basic check: look for opening <table> tags)
    html_tables = 0
    for elem in parsed_report.elements:
        if elem.element_type == 'table' and '<table' in elem.content.lower():
            html_tables += 1
    
    table_figure_stats.append({
        'Company': company,
        'Tables': tables,
        'HTML Tables': html_tables,
        'Figures': figures,
    })

tf_df = pd.DataFrame(table_figure_stats)
print("Tables and Figures Audit:")
print(tf_df.to_string(index=False))

## Sample Structured Elements

In [ ]:
# Show 3 example elements with full metadata
first_company = list(reports.keys())[0]
parsed = reports[first_company]

print(f"Sample elements from {first_company}:")
print()

sample_indices = [0, min(5, len(parsed.elements)-1), min(10, len(parsed.elements)-1)]
for idx in sample_indices:
    if idx < len(parsed.elements):
        elem = parsed.elements[idx]
        print(f"Element {idx}: {elem.element_type.upper()}")
        print(f"  Page: {elem.page_number}, Heading Level: {elem.heading_level}")
        print(f"  Header Hierarchy: {' > '.join(elem.header_hierarchy[:3]) if elem.header_hierarchy else 'None'}")
        content_preview = elem.content[:150].replace('\n', ' ') + '...' if len(elem.content) > 150 else elem.content
        print(f"  Content: {content_preview}")
        print()

## Save Audit Summary

In [ ]:
# Create comprehensive audit summary
audit_summary = []

for company, parsed_report in reports.items():
    type_counts = defaultdict(int)
    with_page_info = 0
    max_heading = 0
    tables_count = 0
    figures_count = 0
    
    for elem in parsed_report.elements:
        type_counts[elem.element_type] += 1
        if elem.page_number >= 0:
            with_page_info += 1
        if elem.element_type == 'heading' and elem.heading_level > 0:
            max_heading = max(max_heading, elem.heading_level)
        if elem.element_type == 'table':
            tables_count += 1
        if elem.element_type == 'figure':
            figures_count += 1
    
    total_elems = len(parsed_report.elements)
    
    audit_summary.append({
        'Company': company,
        'Display Name': COMPANY_MAP.get(company, company),
        'Total Elements': total_elems,
        'Total Pages': parsed_report.total_pages,
        'Elements with Page Info': with_page_info,
        'Page Coverage %': round(with_page_info / total_elems * 100, 1) if total_elems > 0 else 0,
        'Max Heading Level': max_heading,
        'Headings': type_counts['heading'],
        'Paragraphs': type_counts['paragraph'],
        'Tables': tables_count,
        'Figures': figures_count,
        'Lists': type_counts['list'],
    })

audit_df = pd.DataFrame(audit_summary)
output_path = ARTIFACTS_DIR / "report_audit_summary.csv"
audit_df.to_csv(output_path, index=False)

print(f"Audit summary saved to: {output_path}")
print("\nAudit Summary (first 5 rows):")
print(audit_df.head().to_string(index=False))